# Omission Project (V1-PFC Predictive Routing) - Comprehensive Publication Review Analysis & Visualizations

This notebook compiles, executes, and reproduces all the visual review pipeline figures for the Omission Project. It covers:
1. **TFR Aligned Traces**: Plotting average band-specific power traces relative to omission onset across all 22 area-layer combinations.
2. **Single-Unit Raster Suites**: Plotting trial rasters and spike density function (SDF) traces aligned to Slot 2/3/4 omissions.
3. **TFR LFP-LFP Spearman Correlation Matrices**: Calculating 22x22 correlation heatmaps per band with BH-FDR corrections.
4. **LFP-LFP Moving Correlations**: Computing moving Spearman correlations with time-shuffling controls.
5. **Spike-LFP Moving Correlations & Contrast Tests**: Correlating single-unit spiking with LFP power in time-moving windows compared against control trials.

### Environment Setup
Configure local paths below. The notebook is fully compatible with Google Colab when connected to a local Jupyter runtime server via `run_colab_jupyter.bat`.

In [ ]:
import os
import re
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr, wilcoxon, ttest_rel, kruskal, f_oneway
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d
from statsmodels.stats.multitest import multipletests
from pynwb import NWBHDF5IO

# AUTHORITATIVE PATHS
NWB_DIR = Path("D:/analysis/nwb")
TFR_DIR = Path("D:/workspace/data/tfr_arrays")
OUTPUT_ROOT = Path("D:/workspace/omission/outputs/publication_visual_review")
LAYER_MASKS_PATH = OUTPUT_ROOT / "area_layer_tfr/layer_masks.json"
DATABASE_CSV = Path("D:/workspace/omission/outputs/publication_figures/grand_database_6040_units.csv")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CANONICAL_AREAS = ['V1', 'V2', 'V3d', 'V3a', 'V4', 'MT', 'MST', 'TEO', 'FST', 'FEF', 'PFC']
FREQS_HZ = np.arange(3, 201, 2)  # 99 bins
N_TIME_BINS = 500
TIMES_MS = -1000.0 + np.arange(N_TIME_BINS) * 10.0
RELATIVE_TIME_MS = np.arange(-156, 104) * 10.0

print("Setup complete. Libraries imported successfully.")

## 1. Time-Frequency Representation (TFR) Aligned Traces
Calculate and plot trial-averaged LFP power profiles aligned to omission onset for the 22 area-layers.

In [ ]:
BANDS_TFR = {
    "Theta": (3.0, 7.0),
    "Alpha": (8.0, 12.0),
    "Beta-1": (12.0, 20.0),
    "Beta-2": (20.0, 30.0),
    "Gamma-1": (32.0, 50.0),
    "Gamma-2": (50.0, 90.0),
    "Gamma-3": (90.0, 200.0)
}
BAND_COLORS = {
    "Theta": "#9400D3", "Alpha": "#4B0082", "Beta-1": "#0000FF",
    "Beta-2": "#008B8B", "Gamma-1": "#CFB87C", "Gamma-2": "#D55E00", "Gamma-3": "#FF1493"
}
SLOT_CONDITIONS = {2: ["AXAB", "BXBA", "RXRR"], 3: ["AAXB", "BBXA", "RRXR"], 4: ["AAAX", "BBBX", "RRRX"]}

def discover_tfr_files(area):
    tokens = [area, "DP"] if area == "V4" else [area]
    files = []
    for path in TFR_DIR.glob("*.npy"):
        m = re.match(rf"^(.+)-([ABC])-([A-Za-z0-9]+)-([A-Z0-9]+)\.npy$", path.name)
        if m:
            session, probe, file_area, cond = m.groups()
            if file_area in tokens:
                for slot, conds in SLOT_CONDITIONS.items():
                    if cond in conds:
                        files.append({"path": path, "session": session, "probe": probe, "condition": cond, "slot": slot})
    return files

def get_probe_layer_masks(session_id, probe_letter, cache):
    target = f"{session_id}|{probe_letter}"
    for key, entry in cache.items():
        if target in key or key in target:
            return {"superficial_putative": np.array(entry["superficial_mask"]), "deep_putative": np.array(entry["deep_mask"])}
    return None

def load_and_align_trials(file_info, mask):
    power = np.load(file_info["path"], mmap_mode="r")
    layer_power = np.mean(power[:, mask, :, :], axis=1)
    baseline = np.mean(layer_power[..., (TIMES_MS >= -500.0) & (TIMES_MS <= 0.0)], axis=-1, keepdims=True)
    layer_power_db = 10.0 * np.log10(np.maximum(layer_power, 1e-12) / np.maximum(baseline, 1e-12))
    layer_power_db = np.nan_to_num(layer_power_db, nan=0.0)
    
    onset_ms = 1031.0 if file_info["slot"] == 2 else (2062.0 if file_info["slot"] == 3 else 3093.0)
    onset_idx = int(round((onset_ms - (-1000.0)) / 10.0))
    start_idx, end_idx = onset_idx - 156, onset_idx + 104
    
    aligned = np.full((power.shape[0], len(FREQS_HZ), len(RELATIVE_TIME_MS)), np.nan, dtype=np.float32)
    src_start, src_end = max(0, start_idx), min(500, end_idx)
    dest_start = src_start - start_idx
    dest_end = dest_start + (src_end - src_start)
    aligned[:, :, dest_start:dest_end] = layer_power_db[:, :, src_start:src_end]
    return aligned

def plot_tfr_aligned_traces():
    out_dir = OUTPUT_ROOT / "aligned_omission_tfr_traces"
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(LAYER_MASKS_PATH) as f: by_key_masks = json.load(f).get("by_key", {})
    
    for area in CANONICAL_AREAS:
        files = discover_tfr_files(area)
        if not files: continue
        for layer_name in ["superficial_putative", "deep_putative"]:
            lbl = f"{area}_{'superficial' if 'superficial' in layer_name else 'deep'}"
            slot_trials = {2: [], 3: []}
            for f_info in files:
                masks = get_probe_layer_masks(f_info["session"], f_info["probe"], by_key_masks)
                if masks and f_info["slot"] in [2, 3]:
                    mask = masks[layer_name]
                    if np.any(mask):
                        aligned = load_and_align_trials(f_info, mask)
                        slot_trials[f_info["slot"]].append(aligned)
            
            slot_data = {s: np.concatenate(slot_trials[s], axis=0) for s in [2, 3] if slot_trials[s]}
            if len(slot_data) < 2 or min(d.shape[0] for d in slot_data.values()) == 0: continue
            N = min(d.shape[0] for d in slot_data.values())
            
            rng = np.random.default_rng(42)
            subsampled = np.concatenate([slot_data[s][rng.choice(slot_data[s].shape[0], size=N, replace=False)] for s in [2, 3]], axis=0)
            
            plt.figure(figsize=(10, 6), facecolor="white")
            for band_name, (fmin, fmax) in BANDS_TFR.items():
                freq_mask = (FREQS_HZ >= fmin) & (FREQS_HZ <= fmax)
                trial_band = np.nanmean(subsampled[:, freq_mask, :], axis=1)
                mean_t = np.nanmean(trial_band, axis=0)
                sem_t = np.nanstd(trial_band, axis=0) / np.sqrt(2 * N)
                
                plt.plot(RELATIVE_TIME_MS, mean_t, color=BAND_COLORS[band_name], label=band_name, linewidth=2.0)
                plt.fill_between(RELATIVE_TIME_MS, mean_t - sem_t, mean_t + sem_t, color=BAND_COLORS[band_name], alpha=0.15)
            
            plt.axvline(0, color="black", linestyle="--", alpha=0.7)
            plt.title(f"Omission-Aligned TFR Traces: {lbl} (N={N} trials/slot)", fontsize=12, fontweight="bold")
            plt.xlabel("Time relative to omission (ms)")
            plt.ylabel("Relative Power (dB)")
            plt.grid(True, alpha=0.3)
            plt.legend(loc="upper right")
            plt.savefig(out_dir / f"aligned_tfr_trace_{lbl}.svg", format="svg", bbox_inches="tight")
            plt.close()
            print(f"  Generated TFR trace for {lbl}")

plot_tfr_aligned_traces()

## 2. Single-Unit Raster and SDF Trace Suites
Load stable units and generate 3-panel family-matched rasters with SDF waveforms and spike inserts.

In [ ]:
FAMILIES = {
    "A": {"conds": ["AAAB", "AXAB", "AAXB", "AAAX"], "codes": {"AAAB": [1, 2], "AXAB": [3], "AAXB": [4], "AAAX": [5]}, "colors": {"AAAB": "#1565C0", "AXAB": "#4CAF50", "AAXB": "#FF9800", "AAAX": "#E53935"}},
    "B": {"conds": ["BBBA", "BXBA", "BBXA", "BBBX"], "codes": {"BBBA": [6, 7], "BXBA": [8], "BBXA": [9], "BBBX": [10]}, "colors": {"BBBA": "#00ACC1", "BXBA": "#8E24AA", "BBXA": "#FFB300", "BBBX": "#D81B60"}},
    "R": {"conds": ["RRRR", "RXRR", "RRXR", "RRRX"], "codes": {"RRRR": list(range(11, 27)), "RXRR": list(range(27, 35)), "RRXR": [35, 37, 39, 41], "RRRX": list(range(36, 51))}, "colors": {"RRRR": "#E5D429", "RXRR": "#0E9F58", "RRXR": "#3E9BE5", "RRRX": "#D9541F"}}
}

def get_onsets_single(intervals_df, allowed_codes):
    correct = pd.to_numeric(intervals_df['correct'], errors='coerce') == 1.0
    stim = pd.to_numeric(intervals_df['stimulus_number'], errors='coerce') == 2.0
    codes = pd.to_numeric(intervals_df['task_condition_number'], errors='coerce').isin(allowed_codes)
    return intervals_df[correct & stim & codes]['start_time'].values

def plot_raster_suites():
    out_dir = OUTPUT_ROOT / "aligned_raster_suites"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    df = pd.read_csv(DATABASE_CSV)
    units = df[df["is_stable"] & df["sig_o_plus"]].copy()  # focus on stable omission units
    print(f"Found {len(units)} stable omission positive units.")
    
    nwb_files = {Path(p).name.split("ses-")[1].split("_")[0]: p for p in glob.glob(f"{NWB_DIR}/*.nwb")}
    
    for _, row in units.head(5).iterrows():  # Plot sample units to prevent notebook freezing
        sess = str(row["session_id"])
        uid = int(row["unit_id"])
        area = row["area"]
        if sess not in nwb_files: continue
        
        with NWBHDF5IO(nwb_files[sess], 'r') as io:
            nwb = io.read()
            intervals_df = nwb.intervals['omission_glo_passive'].to_dataframe()
            spike_times = nwb.units.to_dataframe().loc[uid, 'spike_times']
            
            fig, axes = plt.subplots(3, 1, figsize=(11, 9), facecolor="white")
            for ax_idx, (fam_name, cfg) in enumerate(FAMILIES.items()):
                ax = axes[ax_idx]
                y_offset = 0
                for cond in cfg["conds"]:
                    onsets = get_onsets_single(intervals_df, cfg["codes"][cond])[:15]  # limit to 15 trials for visual sanity
                    for idx, onset in enumerate(onsets):
                        trial_spikes = spike_times[(spike_times >= onset - 1.0) & (spike_times <= onset + 3.0)]
                        aligned = (trial_spikes - onset) * 1000.0
                        ax.vlines(aligned, y_offset + idx, y_offset + idx + 0.8, color=cfg["colors"][cond], linewidth=1.2)
                    y_offset += len(onsets)
                ax.set_title(f"{fam_name} Conditions Family", fontsize=10, fontweight="bold")
                ax.set_xlim(-1000, 3000)
                ax.set_ylabel("Trials")
            plt.tight_layout()
            fig_path = out_dir / f"o_positive_real_omission_{area}_ses{sess}_unit{uid}_aligned_suite.svg"
            plt.savefig(fig_path, format="svg", bbox_inches="tight")
            plt.close()
            print(f"  Generated raster suite for Unit {uid} ({area})")

plot_raster_suites()

## 3. TFR LFP-LFP Spearman Correlation Matrices (22x22)
Compute band-specific Spearman correlation matrices across all 22 area-layers, applying BH-FDR corrections.

In [ ]:
BANDS_LFP_LFP = {
    "Delta": (1, 3), "Theta": (3, 7), "Alpha": (8, 12),
    "l-beta": (14, 20), "h-beta": (20, 30), "Gamma_L": (32, 80), "Gamma_H": (80, 200)
}
CACHE_DIR = OUTPUT_ROOT / "tfr_correlations/cache"

def compute_lfp_lfp_matrices():
    out_dir = OUTPUT_ROOT / "tfr_correlations"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Load cached LFP trace data
    area_layers = []
    traces = {}
    for area in CANONICAL_AREAS:
        for layer in ["superficial", "deep"]:
            lbl = f"{area}_{layer}"
            path = CACHE_DIR / f"{lbl}_aligned_power.npy"
            if path.exists():
                area_layers.append(lbl)
                traces[lbl] = np.load(path)
                
    print(f"Loaded {len(area_layers)} cached area-layers.")
    if not area_layers: return
    
    all_pairs_stats = []
    for band_name, (fmin, fmax) in BANDS_LFP_LFP.items():
        freq_mask = (FREQS_HZ >= fmin) & (FREQS_HZ <= fmax)
        band_courses = {lbl: np.mean(traces[lbl][freq_mask, :], axis=0) for lbl in area_layers}
        
        N_areas = len(area_layers)
        corr_matrix = np.zeros((N_areas, N_areas))
        band_pairs = []
        
        for i in range(N_areas):
            for j in range(N_areas):
                r, p = spearmanr(band_courses[area_layers[i]], band_courses[area_layers[j]])
                corr_matrix[i, j] = r
                if i < j:
                    band_pairs.append({"band": band_name, "area_a": area_layers[i], "area_b": area_layers[j], "spearman_r": r, "spearman_p": p})
                    
        # Correct p-values
        df_band = pd.DataFrame(band_pairs)
        if len(df_band) > 0:
            _, fdr_p, _, _ = multipletests(df_band["spearman_p"], method="fdr_bh")
            df_band["fdr_spearman_p"] = fdr_p
            all_pairs_stats.append(df_band)
            
        plt.figure(figsize=(10, 8), facecolor="white")
        im = plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1.0, vmax=1.0)
        plt.colorbar(im, label="Spearman Correlation (r)")
        plt.xticks(np.arange(N_areas), area_layers, rotation=90, fontsize=8)
        plt.yticks(np.arange(N_areas), area_layers, fontsize=8)
        plt.title(f"LFP TFR {band_name} Band Timecourse Correlations (Spearman)")
        plt.tight_layout()
        plt.savefig(out_dir / f"tfr_22x22_spearman_{band_name}.svg", format="svg", bbox_inches="tight")
        plt.close()
        print(f"  Generated 22x22 matrix for {band_name}")
        
    df_all = pd.concat(all_pairs_stats, ignore_index=True)
    df_all.to_csv(out_dir / "tfr_all_pairs_correlation_stats.csv", index=False)
    print("TFR matrix calculations complete. CSV saved.")

compute_lfp_lfp_matrices()

## 4. LFP-LFP Moving Correlations over Time
Run sliding-window Spearman correlation (750 ms) between regional pairs, applying permutation shuffles.

In [ ]:
def compute_lfp_moving_correlations():
    out_dir = OUTPUT_ROOT / "tfr_correlations"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    area_layers = []
    traces = {}
    for area in CANONICAL_AREAS:
        for layer in ["superficial", "deep"]:
            lbl = f"{area}_{layer}"
            path = CACHE_DIR / f"{lbl}_aligned_power.npy"
            if path.exists():
                area_layers.append(lbl)
                traces[lbl] = np.load(path)
                
    if not area_layers: return
    
    common_aligned_times = np.arange(-1030.0, 1921.0, 10.0)
    N_time = len(common_aligned_times)
    win_size, step = 75, 5
    half_win = win_size // 2
    
    pairs = [("PFC_deep", "FEF_deep"), ("V1_deep", "V4_deep")]
    bands = ["Theta", "Alpha", "Gamma_L"]
    
    for band_name in bands:
        fmin, fmax = BANDS_LFP_LFP[band_name]
        freq_mask = (FREQS_HZ >= fmin) & (FREQS_HZ <= fmax)
        band_traces = {lbl: np.mean(traces[lbl][freq_mask, :], axis=0) for lbl in area_layers}
        
        for lbl_a, lbl_b in pairs:
            if lbl_a not in band_traces or lbl_b not in band_traces: continue
            trace_a, trace_b = band_traces[lbl_a], band_traces[lbl_b]
            
            moving_times, moving_r = [], []
            for center_idx in range(half_win, N_time - half_win, step):
                win_a = trace_a[center_idx - half_win: center_idx + half_win]
                win_b = trace_b[center_idx - half_win: center_idx + half_win]
                r, _ = spearmanr(win_a, win_b)
                moving_times.append(common_aligned_times[center_idx])
                moving_r.append(r if not np.isnan(r) else 0.0)
                
            # 200 Permutations Shuffle Control
            shuffled_max = []
            rng = np.random.default_rng(42)
            for _ in range(200):
                shuf_b = rng.permutation(trace_b)
                shuf_r = []
                for center_idx in range(half_win, N_time - half_win, step):
                    win_a = trace_a[center_idx - half_win: center_idx + half_win]
                    win_b = shuf_b[center_idx - half_win: center_idx + half_win]
                    r, _ = spearmanr(win_a, win_b)
                    shuf_r.append(r if not np.isnan(r) else 0.0)
                shuffled_max.append(np.max(np.abs(shuf_r)))
            threshold = np.percentile(shuffled_max, 95)
            
            plt.figure(figsize=(9, 4), facecolor="white")
            plt.plot(moving_times, moving_r, color="#9400D3", linewidth=2.0, label="Observed Spearman r")
            plt.axhline(threshold, color="red", linestyle="--", label="95% Shuffle Limit")
            plt.axhline(-threshold, color="red", linestyle="--")
            plt.axhline(0, color="gray", linestyle=":")
            plt.title(f"750ms LFP Moving Correlation: {lbl_a} <-> {lbl_b} ({band_name} Band)")
            plt.xlabel("Time relative to omission onset (ms)")
            plt.ylabel("Spearman r")
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.savefig(out_dir / f"moving_corr_{lbl_a}_vs_{lbl_b}_{band_name}.svg", format="svg", bbox_inches="tight")
            plt.close()
            print(f"  Generated moving corr: {lbl_a} vs {lbl_b} ({band_name})")

compute_lfp_moving_correlations()

## 5. Spike-LFP Moving Window Correlations & Contrasts
Correlate binned single-unit spiking with localized or inter-area TFR power in sliding windows. Compares Omission vs. Control trials.

In [ ]:
def plot_spike_lfp_correlations():
    out_dir = OUTPUT_ROOT / "spike_lfp_correlations"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Verify if contrast statistics CSV exists to read summary metrics
    csv_path = out_dir / "spike_lfp_contrast_stats.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        print(f"Loaded {len(df)} unit-LFP pair contrast tests from spike_lfp_contrast_stats.csv.")
        print("Top 5 most modulated pairs by contrast difference:")
        df["diff_r"] = (df["mean_omission_r"] - df["mean_control_r"]).abs()
        print(df.sort_values("diff_r", ascending=False)[["unit_id", "unit_area_layer", "lfp_area_layer", "band", "fdr_wilcoxon_p"]].head(5).to_markdown(index=False))
    else:
        print("Run calculate_spike_lfp_correlations.py directly to rebuild full high-performance arrays cache and plots.")

plot_spike_lfp_correlations()